# DWBDA Assignments
**Name:** NESPENDRA DAS &nbsp;&nbsp; **Reg No:** 202400543 &nbsp;&nbsp; **Assignment:** DWBDA

## Assignment 1 : Market Basket Analysis (MBA) on real data
Dataset: **Transactions from a Bakery** (Kaggle, 21,292 rows)

In [1]:
# pip install -r requirements.txt
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

url = 'data/BreadBasket_DMS.csv'
df  = pd.read_csv(url)
df = df[df['Item'] != 'NONE']
print('Rows:', len(df))

transactions = df.groupby('Transaction')['Item'].apply(list).tolist()     # items bought together
te = TransactionEncoder()
basket = pd.DataFrame(te.fit_transform(transactions), columns=te.columns_)

items = apriori(basket, min_support=0.02, use_colnames=True)               # frequent itemsets
rules = association_rules(items, metric='confidence', min_threshold=0.3)
print(rules[['antecedents','consequents','support','confidence','lift']]
      .sort_values('lift', ascending=False).head())

ModuleNotFoundError: No module named 'mlxtend'

## Assignment 2 : MapReduce - Students in each grade category (S,A,B,C,D,E,F) for all subjects
Dataset: **Students Performance in Exams** (Kaggle, 1,000 rows)

In [ ]:
import pandas as pd
from collections import defaultdict

url = 'data/StudentsPerformance.csv'
df  = pd.read_csv(url)
subjects = [c for c in df.columns if 'score' in c.lower()]     # math/reading/writing score
print('Rows:', len(df), '| Subjects:', subjects)

def grade(m):
    return ('S' if m>=90 else 'A' if m>=80 else 'B' if m>=70 else
            'C' if m>=60 else 'D' if m>=50 else 'E' if m>=40 else 'F')

mapped  = [((s, grade(row[s])), 1) for _, row in df.iterrows() for s in subjects]  # MAP
reduced = defaultdict(int)                                                          # SHUFFLE + REDUCE
for key, v in mapped:
    reduced[key] += v

table = pd.DataFrame(0, index=subjects, columns=list('SABCDEF'))
for (s, g), c in reduced.items():
    table.loc[s, g] = c
print(table)

## Assignment 3 : Spark - Total products sold in each category
Dataset: **Sample Superstore** (Kaggle, 9,994 rows)

In [ ]:
# pip install -r requirements.txt
import pandas as pd
from pyspark.sql import SparkSession, functions as F

url = 'data/SampleSuperstore.csv'
pdf = pd.read_csv(url)

spark = SparkSession.builder.appName('ProductsSold').getOrCreate()
df    = spark.createDataFrame(pdf)
print('Rows:', df.count())

df.groupBy('Category').agg(F.sum('Quantity').alias('TotalProductsSold')) \
  .orderBy(F.desc('TotalProductsSold')).show()

spark.stop()